# Prov-GigaPath features for colon_n and colon_aca (Colab, T4)
Runs the **unmodified** upstream `extract_features.py` (GeorgeBatch/LC25000-clean @ 912f342) for
`prov-gigapath`, `imagenet` normalisation, fp32, once per class in `CLASSES` (cell 1, default `['colon_n', 'colon_aca']`).
Press *Runtime > Run all* on a T4 GPU runtime.
Needs: `MyDrive/btp_repro/<cls>_original.zip` for each class, Colab secret `HF_TOKEN` (with access to prov-gigapath/prov-gigapath).
Output: `MyDrive/btp_repro/gigapath_out/<cls>/{features.npy,ids.npy,ids_2_img_paths.json}`.
Safe to re-run after a disconnect: a class whose output folder already exists on Drive is skipped. A class folder only appears on Drive after all three files are copied and verified.

In [ ]:
# 1. Config, GPU check, Drive
CLASSES = ['colon_n', 'colon_aca']   # classes to process; each needs MyDrive/btp_repro/<cls>_original.zip
BATCH_SIZE = 32                      # lower to 16 if CUDA runs out of memory
EXPECTED_ZIP_SHA256 = {              # sha256 of data/<cls>_original.zip on the local machine
    'colon_n': '14724d148ccd08631c8977a91e4ef1d5871e959831d29997832ba15bc240169d',
    'colon_aca': '830207449b325b5ae80b8ac6575f182df518c58210ac8a9eabe155f3ef31789d',
}
unknown = [c for c in CLASSES if c not in EXPECTED_ZIP_SHA256]
assert not unknown, f"no expected zip hash for {unknown}"

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
import os
BTP = '/content/drive/MyDrive/btp_repro'
OUT_ROOT = f'{BTP}/gigapath_out'
print("classes:", CLASSES, "| batch size:", BATCH_SIZE)

In [ ]:
# 2. HF token from Colab secrets (never printed)
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], "HF_TOKEN secret is empty (enable 'Notebook access' for it)"
from huggingface_hub import HfApi
HfApi(token=os.environ['HF_TOKEN']).model_info('prov-gigapath/prov-gigapath')   # raises if no access
print("HF access to prov-gigapath/prov-gigapath: OK")

In [ ]:
# 3. Clone upstream at the pinned commit
COMMIT = '912f342fe2a781e847f395d79982dad4d3d425ae'
!rm -rf /content/LC25000-clean
!git clone -q https://github.com/GeorgeBatch/LC25000-clean /content/LC25000-clean
!git -C /content/LC25000-clean checkout -q {COMMIT}
import subprocess
head = subprocess.check_output(['git','-C','/content/LC25000-clean','rev-parse','HEAD']).decode().strip()
assert head == COMMIT, head
print("upstream HEAD:", head)
%cd /content/LC25000-clean

In [ ]:
# 4. Dependencies (keep Colab's torch/torchvision; timm>=1.0.3 is needed for Prov-GigaPath)
!pip install -q "timm>=1.0.3" huggingface_hub safetensors
import torch, torchvision, timm
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__)
assert tuple(int(x) for x in timm.__version__.split('.')[:3]) >= (1, 0, 3)

In [ ]:
# 5. Helpers: one class = unzip, extract with the unmodified upstream script, check, copy to Drive
import glob, hashlib, json, shutil, subprocess, sys
import numpy as np

UP = '/content/LC25000-clean'
FILES = ['features.npy', 'ids.npy', 'ids_2_img_paths.json']


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def run_streaming(cmd):
    """Run cmd in the upstream repo, print its output live, raise on a non-zero exit.
    stdin is closed so upstream can never block on its overwrite prompt."""
    proc = subprocess.Popen(cmd, cwd=UP, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip())
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


def check_outputs(cls, folder):
    feats = np.load(f'{folder}/features.npy')
    assert feats.shape == (5000, 1536), feats.shape
    assert np.isfinite(feats).all(), "non-finite values in features"
    ids = np.load(f'{folder}/ids.npy')
    assert np.array_equal(ids, np.arange(5000)), "ids.npy is not 0..4999"
    ids2p = json.load(open(f'{folder}/ids_2_img_paths.json'))
    assert sorted(map(int, ids2p)) == list(range(5000)), "ids_2_img_paths keys are not 0..4999"
    assert len(set(ids2p.values())) == 5000, "duplicate image paths"
    prefix = f'./LC25000/{cls}/'
    assert all(p.startswith(prefix) for p in ids2p.values()), f"paths do not start with {prefix}"


def report(cls, folder, fresh):
    feats = np.load(f'{folder}/features.npy')
    print(f"[{cls}] features: {feats.shape} {feats.dtype} | finite: {bool(np.isfinite(feats).all())}")
    if fresh:
        print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| timm", timm.__version__)
    else:
        print("(extracted in an earlier run; versions of that run are not known to this runtime)")
    for f in FILES:
        print(sha256_of(f'{folder}/{f}'), f'{cls}/{f}')


def process_class(cls):
    out = f'{OUT_ROOT}/{cls}'
    print(f"\n===== {cls} =====")
    if all(os.path.isfile(f'{out}/{f}') for f in FILES):
        print(f"outputs already on Drive ({out}): skipping extraction")
        report(cls, out, fresh=False)
        return

    zip_path = f'{BTP}/{cls}_original.zip'
    assert os.path.isfile(zip_path), f"missing {zip_path}"
    print("zip bytes:", os.path.getsize(zip_path))
    digest = sha256_of(zip_path)
    print("zip sha256:", digest)
    assert digest == EXPECTED_ZIP_SHA256[cls], f"{zip_path}: sha256 {digest} != expected {EXPECTED_ZIP_SHA256[cls]}"

    img_dir = f'{UP}/LC25000/{cls}'
    shutil.rmtree(img_dir, ignore_errors=True)
    os.makedirs(img_dir)
    subprocess.run(['unzip', '-q', '-o', zip_path, '-d', img_dir], check=True)
    n = len(glob.glob(f'{img_dir}/*.jpeg'))
    assert n == 5000 and len(os.listdir(img_dir)) == 5000, (n, len(os.listdir(img_dir)))
    print("jpeg files:", n)

    # a leftover local result would make upstream ask before overwriting
    fd = f'{UP}/feature_vectors/{cls}/prov-gigapath/imagenet'
    shutil.rmtree(fd, ignore_errors=True)
    run_streaming([sys.executable, 'extract_features.py', '--cancer_type', cls, '--img_norm', 'imagenet',
                   '--extractor_name', 'prov-gigapath', '--device', 'cuda', '--batch_size', str(BATCH_SIZE)])
    check_outputs(cls, fd)

    # copy to <cls>.tmp, verify, then rename: a folder under its final name is always complete
    tmp = out + '.tmp'
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    for f in FILES:
        shutil.copy(f'{fd}/{f}', f'{tmp}/{f}')
        assert sha256_of(f'{fd}/{f}') == sha256_of(f'{tmp}/{f}'), f"copy of {f} differs"
    os.rename(tmp, out)
    print("copied to", out)
    report(cls, out, fresh=True)

In [ ]:
# 6. Extract for every class in CLASSES (a class already on Drive is skipped)
for cls in CLASSES:
    process_class(cls)

In [ ]:
# 7. Final check: every class is on Drive, upstream is still the pinned commit and unmodified
for cls in CLASSES:
    missing = [f for f in FILES if not os.path.isfile(f'{OUT_ROOT}/{cls}/{f}')]
    assert not missing, (cls, missing)
head = subprocess.check_output(['git', '-C', UP, 'rev-parse', 'HEAD']).decode().strip()
dirty = subprocess.check_output(['git', '-C', UP, 'status', '--porcelain']).decode().strip()
assert head == COMMIT and not dirty, (head, dirty)
print("upstream HEAD:", head, "| working tree clean")
print("ALL CLASSES ON DRIVE:", CLASSES, "->", OUT_ROOT)